# Classification Supervisor de Courriels : Spam vs Ham
### Projet de NLP & Machine Learning pour la Détection d'Emails Malveillants

Ce notebook constitue un guide complet et interactif pour concevoir, évaluer et déployer un classifieur de courriels capable de distinguer les messages légitimes (**ham**) des messages indésirables ou malveillants (**spam**).

--- 
### Objectifs du Notebook :
1. **Analyse Exploratoire des Données (EDA)** : Comprendre la distribution des classes et visualiser les mots-clés via des WordClouds.
2. **Pipeline de Prétraitement de Texte (NLP)** : Nettoyage, tokenisation, suppression des stopwords, élimination de la ponctuation et réduction des mots à leur racine (*Stemming*).
3. **Vectorisation numérique** : Transformation du texte brut en caractéristiques numériques grâce à la méthode TF-IDF ($Term\;Frequency\;-\;Inverse\;Document\;Frequency$).
4. **Entraînement de Modèles** : Comparaison de plusieurs algorithmes de classification (Naive Bayes, Régression Logistique, SVM, Random Forest).
5. **Évaluation** : Analyse détaillée des performances à l'aide de matrices de confusion et de rapports de classification (Précision, Rappel, F1-Score).

## Étape 0 : Installation et Configuration de l'Environnement
Avant de commencer, installons les bibliothèques indispensables si elles ne sont pas déjà présentes dans votre environnement.

In [ ]:
# Installation des packages nécessaires
!pip install -q pandas numpy nltk scikit-learn wordcloud matplotlib seaborn

In [2]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Outils de NLP (NLTK)
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Téléchargement des ressources NLTK nécessaires
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("Configuration initiale terminée avec succès.")

Configuration initiale terminée avec succès.


## Étape 1 : Analyse des Données & Génération du Dataset
Afin de rendre ce notebook directement fonctionnel sans nécessiter le téléchargement préalable d'un fichier externe, nous allons générer un dataset synthétique réaliste contenant des exemples d'emails légitimes et d'emails de phishing/spam.

In [3]:
# Création d'un dataset synthétique robuste pour la démonstration
raw_data = {
    'text': [
        # Emails Légitimes (Ham)
        "Hi team, please find attached the minutes of our last project review meeting. Let me know if you have questions.",
        "Are we still on for lunch today? I was thinking about that small Italian restaurant near the office.",
        "Dear candidate, thank you for submitting your application for the ML Engineer position. We will review it shortly.",
        "Please review the updated code on Github and approve the pull request before our daily standup.",
        "Can you send me the Excel spreadsheet with the budget projection for next quarter? Thanks, Sarah.",
        "Hey, don't forget we have a birthday celebration in the main cafeteria at 4 PM.",
        "The server maintenance is scheduled for Sunday morning. Expect minor downtime between 2 AM and 5 AM.",
        "Thanks for your help yesterday, the database connection issue is now completely resolved.",
        # Emails Malveillants / Spams
        "CONGRATULATIONS! You have been selected to win a free iPhone 15. Click here now to claim your prize!",
        "URGENT: Your bank account has been suspended due to suspicious activity. Log in immediately to verify identity.",
        "Get rich quick! Work from home and earn up to $5000 a week. No experience required. Register at this link!",
        "Dear Customer, your invoice is overdue. Please open the attached zip archive to check the transaction details.",
        "Exclusive offer! Buy cheap medication online without prescription. High quality guaranteed. Fast shipping!",
        "You won the lottery! To claim your cash prize of $1,000,000, send your credit card details immediately.",
        "Verify your password now or your account will be permanently deactivated within 24 hours. Click link below.",
        "Refinance your mortgage today! Unbeatable low interest rates guaranteed. Apply now for free quote!"
    ],
    'label': ['ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 
             'spam', 'spam', 'spam', 'spam', 'spam', 'spam', 'spam', 'spam']
}

df = pd.DataFrame(raw_data)

# Ajout délibéré de doublons et de valeurs manquantes pour la démonstration du nettoyage
df = pd.concat([df, pd.DataFrame({'text': [None, "Are we still on for lunch today? I was thinking about that small Italian restaurant near the office."], 'label': ['ham', 'ham']})], ignore_index=True)

print(f"Taille brute du dataset : {df.shape[0]} lignes.")

Taille brute du dataset : 18 lignes.


### 1.1 Examen de la Structure et Nettoyage

In [4]:
print("--- Aperçu des données --- ")
display(df.head())

print("\n--- Informations sur le Dataset ---")
print(df.info())

print("\n--- Détection des valeurs manquantes ---")
print(df.isnull().sum())

print("\n--- Détection des doublons ---")
print(f"Nombre total de doublons exacts : {df.duplicated().sum()}")

--- Aperçu des données --- 


,text,label
0,"Hi team, please find attached the minutes of o...",ham
1,Are we still on for lunch today? I was thinkin...,ham
2,"Dear candidate, thank you for submitting your ...",ham
3,Please review the updated code on Github and a...,ham
4,Can you send me the Excel spreadsheet with the...,ham



--- Informations sur le Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    17 non-null     object
 1   label   18 non-null     object
dtypes: object(2)
memory usage: 420.0+ bytes
None

--- Détection des valeurs manquantes ---
text     1
label    0
dtype: int64

--- Détection des doublons ---
Nombre total de doublons exacts : 1


In [ ]:
# Suppression des valeurs manquantes dans la colonne d'in


# Suppression des doublons


print(f"Taille après nettoyage des valeurs manquantes et doublons : {df.shape[0]} lignes.")

### 1.2 Distribution de la Variable Cible
Analysons si le dataset est équilibré.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df, palette='Set2')
plt.title("Distribution des classes (Spam vs Ham)")
plt.xlabel("Catégorie")
plt.ylabel("Nombre de messages")
plt.show()

print("Proportions des classes :")
print(df['label'].value_counts(normalize=True) * 100)

### 1.3 Visualisation par WordCloud (Nuage de Mots)
Générons des nuages de mots pour visualiser les termes les plus fréquents caractérisant chaque type d'email.

In [ ]:
spam_text = " ".join(df[df['label'] == 'spam']['text'])
ham_text = " ".join(df[df['label'] == 'ham']['text'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8))

# WordCloud Spam
wordcloud_spam = WordCloud(width=800, height=400, background_color='white', colormap='Reds').generate(spam_text)
ax1.imshow(wordcloud_spam, interpolation='bilinear')
ax1.set_title("Mots fréquents - SPAM", fontsize=16, color='red')
ax1.axis('off')

# WordCloud Ham
wordcloud_ham = WordCloud(width=800, height=400, background_color='white', colormap='GnBu').generate(ham_text)
ax2.imshow(wordcloud_ham, interpolation='bilinear')
ax2.set_title("Mots fréquents - HAM (Légitimes)", fontsize=16, color='green')
ax2.axis('off')

plt.tight_layout()
plt.show()

## Étape 2 : Prétraitement du Texte

Le texte brut contient énormément de bruit (majuscules, ponctuation, mots grammaticaux vides, etc.). Le but du prétraitement est de réduire la dimensionnalité tout en conservant l'information sémantique principale.

Nous allons créer une fonction `preprocess_text` qui réalise les opérations suivantes :
1. **Conversion en minuscules** : Éviter que 'Free' et 'free' soient vus comme deux mots différents.
2. **Suppression de la ponctuation et caractères spéciaux** : Via des expressions régulières (`re`).
3. **Tokenisation** : Découpage des phrases en mots individuels.
4. **Suppression des Stopwords** : Retrait des mots de liaison peu informatifs (*the, to, are, are...*).
5. **Stemming (Racinisation)** : Réduction des mots à leur radical avec l'algorithme `PorterStemmer` (par ex. *connection*, *connected*, *connects* deviennent tous *connect*).

In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Normalisation en minuscules
    
    
    # 2. Suppression de la ponctuation et des caractères spéciaux
    
    
    # 3. Tokenisation
    
    
    # 4. Suppression des stopwords
    
    
    # 5. Stemming (PorterStemmer)
    
    
    # Reconstruction du texte nettoyé
    return " ".join(stemmed_tokens)

# Exemple de démonstration pas-à-pas
sample_email = "CONGRATULATIONS! You have selected to win a free connection, click here!"
print(f"Texte original :  {sample_email}")
print(f"Texte prétraité : {preprocess_text(sample_email)}")

In [ ]:
# Application de la fonction de traitement à l'ensemble de notre colonne textuelle
df['clean_text'] = df['text'].apply(preprocess_text)

print("Aperçu du dataset après prétraitement :")
display(df[['text', 'clean_text', 'label']].head())

## Étape 3 : Vectorisation & Préparation des Données

Les modèles de Machine Learning ne peuvent ingérer que des données numériques. Nous allons utiliser la méthode de pondération **TF-IDF** ($Term\;Frequency\;-Inverse\;Document\;Frequency$) via `TfidfVectorizer()`. 

La formule mathématique de calcul d'un poids TF-IDF pour un terme $t$ dans un document $d$ au sein d'un corpus $D$ est :

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

Où :
- $\text{TF}(t, d)$ est la fréquence du terme $t$ dans le document $d$.
- $\text{IDF}(t, D) = \log\left(\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|}\right) + 1$ (version lissée standard de Scikit-Learn).

In [10]:
# Encodage des étiquettes cibles en valeurs numériques (ham = 0, spam = 1)
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})

X = df['clean_text']
y = df['label_encoded']

# Division du jeu de données en jeu d'entraînement (80%) et de test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Échantillons d'entraînement : {X_train.shape[0]}")
print(f"Échantillons de test : {X_test.shape[0]}")

Échantillons d'entraînement : 12
Échantillons de test : 4


In [ ]:
# Initialisation du vectoriseur TF-IDF


# Ajustement sur les données d'entraînement et transformation


# Transformation uniquement sur les données de test (pour éviter toute fuite de données / data leakage)


print(f"Matrice d'entraînement TF-IDF générée avec {X_train_tfidf.shape[1]} caractéristiques distinctes.")

## Étape 4 : Entraînement et Comparaison des Modèles
Nous allons instancier et tester 4 algorithmes classiques et redoutablement efficaces pour la classification de textes court :
1. **Multinomial Naive Bayes (MNB)** : Algorithme probabiliste basé sur le théorème de Bayes, très adapté aux décomptes de mots.
2. **Régression Logistique (LR)** : Modèle linéaire simple, interprétable et très robuste.
3. **Linear Support Vector Classifier (Linear SVC)** : Cherche l'hyperplan optimal séparant au mieux les spams et hams dans l'espace multidimensionnel.
4. **Random Forest Classifier (RF)** : Ensemble d'arbres de décision pour capter les interactions non-linéaires entre les mots.

In [ ]:
# Dictionnaire des modèles à évaluer
models = {
    "Naive Bayes (Multinomial)": MultinomialNB(),
    "Logistic Regression": LogisticRegression(solver='liblinear'),
    "Linear Support Vector Classifier": LinearSVC(dual=False),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# Dictionnaire pour stocker les scores
results = {}

for name, model in models.items():
    # Entraînement
    
    
    # Prédiction sur l'échantillon de test
    
    
    # Calcul de la justesse (Accuracy)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {
        'accuracy': acc,
        'predictions': y_pred,
        'model_instance': model
    }
    print(f"Modèle '{name}' entraîné avec succès. Accuracy : {acc * 100:.2f}%")

### 4.2 Analyse Détaillée des Métriques

In [ ]:
# Choix du modèle pour un focus détaillé et tracé des matrices de confusion
for name, res in results.items():
    print(f"\n{'='*20} {name} {'='*20}")
    print(classification_report(y_test, res['predictions'], target_names=['Ham', 'Spam'], zero_division=0))
    
    # Tracé de la matrice de confusion
    cm = confusion_matrix(y_test, res['predictions'])
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
    plt.title(f"Matrice de Confusion : {name}")
    plt.ylabel('Classe Réelle')
    plt.xlabel('Classe Prédite')
    plt.show()

## Étape 5 : Module de Prédiction Intégré (Inférence)
Testons notre pipeline de classification sur de tout nouveaux courriels qui n'ont jamais été vus par les modèles.

In [ ]:
def predict_email(model, email_text):
    # 1. Prétraitement linguistique
    processed = preprocess_text(email_text)
    
    # 2. Vectorisation TF-IDF
    features = tfidf_vectorizer.transform([processed])
    
    # 3. Prédiction
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0] if hasattr(model, "predict_proba") else [None, None]
    
    label = "SPAM" if prediction == 1 else "HAM (Légitime)"
    conf = f"(Probabilité: {probability[prediction]*100:.2f}%)" if probability[0] is not None else ""
    
    print(f"Email: '{email_text}'")
    print(f"Résultat de classification : -> {label} {conf}\n")
best_model = results["Logistic Regression"]['model_instance']

new_emails = [
    "Hi John, are you available for the team syncing meeting tomorrow morning at 9?",
    "URGENT!! Claim your free bonus prize right now! Visit our cash link instantly."
]

for mail in new_emails:
    predict_email(best_model, mail)